# GF(8) Toric Code — Canonical Champion Search (Kaggle)

Uses **AGL₂(F₇)** symmetry (order 98,784) to reduce the search space.
Unlike the BFS notebook, this search is **complete**: it explores every
equivalence class of k-subsets, so it cannot miss champions due to good
codes lacking good sub-codes.

The exponents live in F₇² (since GF(8)* ≅ Z₇), so the symmetry group is
AGL₂(F₇) = F₇² ⋊ GL₂(F₇), order 49 × 2016 = **98,784**.

Approximate canonical form counts (no pruning):

| k | raw C(49,k) | canonical forms |
|---|-------------|----------------|
| 7 | 62 M | ~630 |
| 8 | 450 M | ~4,600 |
| 9 | 2.5 B | ~25,000 |
| 10 | 10 B | ~100,000 |

In [ ]:
!git clone https://github.com/t-ravnushkin/generalized-toric-gpu-search.git
import sys, os
sys.path.insert(0, "/kaggle/working/generalized-toric-gpu-search")
os.chdir("/kaggle/working")

In [ ]:
%pip install pyopencl

In [ ]:
import cupy as cp
n = cp.cuda.runtime.getDeviceCount()
for i in range(n):
    p = cp.cuda.runtime.getDeviceProperties(i)
    print(f"Device {i}: {p['name'].decode()}  "
          f"({p.get('multiProcessorCount', 0)} SMs, "
          f"{p['totalGlobalMem'] // 1024**3} GB)")

In [ ]:
from precompute import build_eval_matrix, bounding_cube_points

M       = build_eval_matrix()
lattice = bounding_cube_points()

from kernel_cuda_bp import init_cuda_oracles_bp
oracles, sm_count = init_cuda_oracles_bp(M)
n_gpus = len(oracles)

BATCH_SIZE = sm_count * 1024 * n_gpus
print(f"\n{n_gpus} GPU(s) ready  batch={BATCH_SIZE:,}")

## Codetables.de bounds for [49, k] codes over GF(8)

In [ ]:
from codetables import bounds_for_n
import pandas as pd

targets = bounds_for_n(49)

df = pd.DataFrame(
    [(k, d) for k, d in sorted(targets.items()) if k <= 15],
    columns=["k", "best_known_d"]
)
display(df)

## Canonical search

`prune_margin=None` — complete, no pruning (recommended for k ≤ 10).  
`prune_margin=m`    — keep sets with d ≥ target − m.  
`prune_eval_mode="champion"` — fast record-hunting mode: evaluates at the champion target,
records verified champions, and keeps an optimistic frontier.  
`screen_from_k=12` — sampled screening for deep levels. It keeps the run moving, but records
only unverified candidates from screened levels.

In [ ]:
import time
from pathlib import Path
from datetime import datetime
from champion_search_canonical import canonical_champion_search, find_latest_results

results_file = find_latest_results()
if results_file is None:
    results_file = Path(f"canon_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json")
    print(f"Starting fresh: {results_file}")
else:
    print(f"Resuming: {results_file}")

# ── parameters ──────────────────────────────────────────────────
MAX_K                       = 15
PRUNE_MARGIN                = 6             # keep sets with d >= target - margin; larger = broader search
PRUNE_EVAL_MODE             = "champion"    # "champion" = fast optimistic frontier; "survivor" = exact but slower
K_CANONICAL_MAX             = 15            # k ≤ this: full AGL₂ canonical forms (GPU kernel supports k≤16)
SCREEN_FROM_K               = 12            # from here onward, sample instead of exact-certifying every set
SCREEN_SAMPLE_COUNT         = 2_000_000     # random codewords per set at screened levels; raise for stricter screening
SCREEN_CANDIDATE_LOG_LIMIT  = 200           # human-readable unverified candidates per screened level
# ────────────────────────────────────────────────────────────────

t0 = time.perf_counter()
canonical_champion_search(
    oracles                    = oracles,
    lattice                    = lattice,
    results_file               = results_file,
    targets                    = targets,
    max_k                      = MAX_K,
    batch_size                 = BATCH_SIZE,
    prune_margin               = PRUNE_MARGIN,
    resume                     = True,
    k_canonical_max            = K_CANONICAL_MAX,
    prune_eval_mode            = PRUNE_EVAL_MODE,
    screen_from_k              = SCREEN_FROM_K,
    screen_sample_count        = SCREEN_SAMPLE_COUNT,
    screen_candidate_log_limit = SCREEN_CANDIDATE_LOG_LIMIT,
)
print(f"\nTotal time: {time.perf_counter() - t0:.1f}s")

## Results vs best known bounds

In [ ]:
from champion_search_canonical import load_results

records = [
    r for r in load_results(results_file)
    if r.get("type") not in ("level_complete", "survivor")
    and "k" in r and "min_distance" in r
]

df_res = pd.DataFrame(records)[["name", "k", "min_distance", "best_known_d", "new_record"]]
df_res["gap"] = df_res["best_known_d"] - df_res["min_distance"]
df_res.sort_values(["k", "min_distance"], ascending=[True, False], inplace=True)
df_res.reset_index(drop=True, inplace=True)

print(f"Champions found: {len(df_res)}")
print(f"New records:     {df_res['new_record'].sum()}")
display(df_res)

In [ ]:
# Per-level summary: canonical form counts and champion counts
level_recs = [r for r in load_results(results_file) if r.get("type") == "level_complete"]
df_levels = pd.DataFrame(level_recs)[["k", "n_canonical", "n_champions", "n_survivors"]]
df_levels["reduction_vs_raw"] = df_levels.apply(
    lambda row: f"1/{int(__import__('math').comb(49, row.k)) // max(row.n_canonical, 1):,}x"
    if row.n_canonical > 0 else "", axis=1
)
display(df_levels)